# 03 — Transformer Architecture

Define a minimal decoder-only (GPT-style) transformer for next-token
prediction over the 5-state alphabet.  Walk through the architecture,
count parameters, and verify the forward pass.

In [1]:
import torch
import torch.nn as nn

## Model

Architecture: token embedding → learned positional embedding →
N causal TransformerEncoder layers (pre-norm) → linear head over vocab.

This is a standard GPT-style setup.  The causal attention mask ensures
position *t* only attends to positions 0 … t, so the model predicts the
next token from its causal history — the same information used to define
the Markov chain.

In [ ]:
class MarkovTransformer(nn.Module):
    """Decoder-only transformer for next-token prediction.

    Given an input sequence of length L, the model returns logits of shape
    (batch, L, vocab_size).  The prediction at position t uses only tokens
    0 … t (causal mask).  For next-token prediction, pair output[:,t,:]
    with target token t+1.

    Args:
        vocab_size: Number of discrete tokens (default 5 for Gambler's Ruin).
        d_model: Embedding dimension.
        nhead: Number of attention heads.  Must divide d_model.
        num_layers: Number of TransformerEncoder layers.
        dim_feedforward: Hidden size of the position-wise FFN.
        max_len: Maximum sequence length (for positional embedding table).
    """

    def __init__(
        self,
        vocab_size: int = 5,
        d_model: int = 32,
        nhead: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 64,
        max_len: int = 64,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=0.0,  # no dropout for analysis clarity
            batch_first=True,
            norm_first=True,  # pre-norm: more stable training
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        # Bias-free head: the embedding already carries intercept information
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    # ------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            x: LongTensor (batch, seq_len) of token indices.

        Returns:
            logits: FloatTensor (batch, seq_len, vocab_size).
        """
        B, L = x.shape
        positions = torch.arange(L, device=x.device)
        h = self.token_embedding(x) + self.pos_embedding(positions)

        # Causal (upper-triangular) additive mask.
        # mask[i, j] = 0 if j <= i (can attend), -inf if j > i (future).
        causal_mask = torch.triu(
            torch.full((L, L), float("-inf"), device=x.device), diagonal=1
        )
        h = self.transformer(h, mask=causal_mask, is_causal=True)
        return self.head(h)

    # ------------------------------------------------------------------
    def predict_probs(self, x: torch.Tensor) -> torch.Tensor:
        """Softmax next-token probabilities.  Shape (batch, seq_len, vocab_size)."""
        return self.forward(x).softmax(dim=-1)

## Instantiate & inspect

In [ ]:
# Default config — kept small so the experiment runs on CPU
VOCAB_SIZE = 5
D_MODEL = 32
NHEAD = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 64
MAX_LEN = 64

model = MarkovTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    max_len=MAX_LEN,
)
print(model)

In [ ]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")

# Per-module breakdown
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20} {n:>6,} params")

## Smoke test

In [ ]:
# Verify output shapes for a random batch
batch_size, seq_len = 4, 16
x = torch.randint(0, VOCAB_SIZE, (batch_size, seq_len))
logits = model(x)
print(f"Input  shape: {x.shape}")
print(f"Output shape: {logits.shape}  (expected: {(batch_size, seq_len, VOCAB_SIZE)})")
assert logits.shape == (batch_size, seq_len, VOCAB_SIZE)

# Softmax probabilities at position 0 should sum to 1 for each batch element
probs0 = logits[:, 0, :].softmax(dim=-1)
assert torch.allclose(probs0.sum(dim=-1), torch.ones(batch_size), atol=1e-5)
print("✓ Output shapes and probability sums are correct")